### Задание на Четверку (Проведение исследований моделями семантической сегментации)

#### 2. Создание бейзлайна и оценка качества

Обучим модели используя  segmentation_models.pytorch, но для начала установим необходимые библиотеки:

In [113]:
!pip install segmentation-models-pytorch albumentations opencv-python


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


импорты

In [114]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm

пути к данным

In [115]:
IMAGES_DIR = "../train_images"
MASKS_DIR = "../train_labels"

dataset

In [116]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted([
            f for f in os.listdir(images_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.images)

    def find_mask(self, name):
        for ext in [".png", ".jpg", ".jpeg"]:
            path = os.path.join(self.masks_dir, name + ext)
            if os.path.exists(path):
                return path
        return None

    def __getitem__(self, idx):
        img_name = self.images[idx]
        name = os.path.splitext(img_name)[0]

        img_path = os.path.join(self.images_dir, img_name)
        mask_path = self.find_mask(name)

        if mask_path is None:
            raise ValueError(f"Mask not found for {name}")

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Bad image: {img_path}")
        if mask is None:
            raise ValueError(f"Bad mask: {mask_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = (mask > 0).astype("float32")

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        return image, mask.unsqueeze(0)

аугментации

In [117]:
def get_transforms():
    return A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(),
        ToTensorV2()
    ])

dataLoader

In [118]:
dataset = SegmentationDataset(
    IMAGES_DIR,
    MASKS_DIR,
    transform=get_transforms()
)

train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

ValueError: num_samples should be a positive integer value, but got num_samples=0

CNN (U-Net) модель

In [ ]:
modelCNN = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

Transformer (SegFormer-подобный)

In [ ]:
modelTransformer = smp.Unet(
    encoder_name="mit_b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

Настройка обучения моделей

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelCNN = modelCNN.to(device)
modelTransformer = modelTransformer.to(device)

loss_fn = smp.losses.DiceLoss(mode='binary')
optimizerCNN = torch.optim.Adam(modelCNN.parameters(), lr=1e-3)
optimizerTransformer = torch.optim.Adam(modelTransformer.parameters(), lr=1e-3)

Цикл обучения

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, masks in tqdm(loader):
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Запуск обучения

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    lossCNN = train_one_epoch(modelCNN, train_loader, optimizerCNN)
    print(f"CNN: Epoch {epoch+1}: Loss = {lossCNN:.4f}")
    
    lossTransformer = train_one_epoch(modelTransformer, train_loader, optimizerTransformer)
    print(f"Transformer: Epoch {epoch+1}: Loss = {lossTransformer:.4f}")

  0%|          | 0/1 [00:00<?, ?it/s]


ValueError: Bad image: ../train_images\LICENSE.txt

реализация метрик

In [ ]:
def iou_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    
    return (intersection + 1e-7) / (union + 1e-7)


def dice_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    
    intersection = (pred * target).sum()
    
    return (2 * intersection + 1e-7) / (pred.sum() + target.sum() + 1e-7)


def pixel_accuracy(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    
    correct = (pred == target).float().sum()
    total = torch.numel(pred)
    
    return correct / total

функция оценки модели

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    
    iou_total = 0
    dice_total = 0
    acc_total = 0
    
    with torch.no_grad():
        for images, masks in tqdm(loader):
            images = images.to(device)
            masks = masks.to(device)

            preds = model(images)
            preds = torch.sigmoid(preds)

            iou_total += iou_score(preds, masks).item()
            dice_total += dice_score(preds, masks).item()
            acc_total += pixel_accuracy(preds, masks).item()

    n = len(loader)
    
    return {
        "IoU": iou_total / n,
        "Dice": dice_total / n,
        "Pixel Accuracy": acc_total / n
    }

оценка обеих моделей

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

metrics_cnn = evaluate_model(modelCNN, val_loader)
metrics_transformer = evaluate_model(modelTransformer, val_loader)

вывод результатов

In [ ]:
print("\n=== CNN Metrics ===")
for k, v in metrics_cnn.items():
    print(f"{k}: {v:.4f}")

print("\n=== Transformer Metrics ===")
for k, v in metrics_transformer.items():
    print(f"{k}: {v:.4f}")